# Copy Test - LoRA

Compare the **original Kaggle E4B W4A16 model** against the **same served model with a rank-128 LoRA adapter**, without training or submitting. This tests whether the real competition vLLM stack applies the correction, not just the llama.cpp merged-weight proxy.

**Kaggle setup:** Enable **4× L4 GPUs**, attach the competition **Gemma 4 Developer Agent wheelhouse**, the **`gemma-4-e4b-it-qat-w4a16-ct` model**, and a private Kaggle Dataset containing the generated rank-128 adapter (`adapter_config.json` + `adapter_model.safetensors`) from `experiments/lora_feasibility/adapters/ple_proj_svd_r128/`. The adapter is **not** included in this notebook; attach it before running. Internet can remain off.

Run: **install → locate model/adapter → start LoRA-enabled server → paired copy test → summary**. The notebook does not modify the model, train, or submit.


After running all cells, download `copy_lora_evidence.zip` from the notebook Output tab. It contains the observed paired results, adapter hash, configuration, and vLLM log.


In [1]:
# 1. Install the competition's offline inference wheels (from the supplied notebook).
import glob
import os
import shutil
import subprocess
import sys
from pathlib import Path

for key, value in {
    "TRANSFORMERS_NO_TF": "1",
    "TRANSFORMERS_NO_FLAX": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "VLLM_NO_USAGE_STATS": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "VLLM_ENGINE_READY_TIMEOUT_S": "1200",
    "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1",
    "NCCL_DEBUG": "WARN",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "OTEL_SDK_DISABLED": "true",
    "OTEL_PYTHON_DISABLED": "true",
}.items():
    os.environ[key] = value

wheelhouse = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
assert wheelhouse.is_dir(), f"Attach the competition wheelhouse dataset: {wheelhouse}"

# Preserve the Getting Started notebook's cutlass workaround and wheel-name repair.
for pattern in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth", 
                "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
    for hook in glob.glob(pattern):
        try:
            os.unlink(hook)
        except OSError:
            pass

tmp = Path("/tmp/wheelhouse")
tmp.mkdir(exist_ok=True)
for wheel in wheelhouse.glob("*.whl"):
    if "cutlass" in wheel.name.lower():
        continue
    name = wheel.name.replace("cu128", "+cu128") if "cu128" in wheel.name and "+" not in wheel.name else wheel.name
    dest = tmp / name
    if not dest.exists():
        try:
            dest.symlink_to(wheel)
        except OSError:
            shutil.copyfile(wheel, dest)

wheels = sorted(map(str, tmp.glob("*.whl")))
assert wheels, f"No wheel files in {wheelhouse}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", *wheels], check=True)
print(f"Installed {len(wheels)} competition wheels.")

Installed 41 competition wheels.


In [2]:
# 2. Locate the E4B competition model and the ATTACHED rank-128 adapter dataset.
import json
from pathlib import Path

MODEL_NAME = "gemma-4-e4b-it-qat-w4a16-ct"
MODEL_PATH = Path(f"/kaggle/input/models/google/gemma-4/other/{MODEL_NAME}/2")
assert MODEL_PATH.is_dir(), f"Attach the E4B model at {MODEL_PATH}"

# Optional override if your dataset includes multiple adapters.
ADAPTER_DIR_OVERRIDE = "/kaggle/input/datasets/michi883/ple-proj-svd-r128"  # e.g. /kaggle/input/my-copy-lora/ple_proj_svd_r128
if ADAPTER_DIR_OVERRIDE:
    ADAPTER_PATH = Path(ADAPTER_DIR_OVERRIDE)
else:
    candidates = [p.parent for p in Path("/kaggle/input").rglob("adapter_model.safetensors")
                  if (p.parent / "adapter_config.json").exists()]
    assert candidates, ("No adapter_model.safetensors + adapter_config.json found. "
                        "Attach the rank-128 adapter as a Kaggle Dataset.")
    ranked = [p for p in candidates if 'ple_proj_svd_r128' in str(p)]
    assert len(ranked) == 1 or (not ranked and len(candidates) == 1), (
        "Multiple adapters found: " + repr([str(p) for p in candidates]) +
        ". Set ADAPTER_DIR_OVERRIDE explicitly.")
    ADAPTER_PATH = (ranked or candidates)[0]

config = json.loads((ADAPTER_PATH / "adapter_config.json").read_text())
rank = int(config.get("r", -1))
assert rank == 128, f"Expected rank-128 adapter, got rank={rank}"
assert (ADAPTER_PATH / 'adapter_model.safetensors').is_file()
assert (ADAPTER_PATH / 'adapter_config.json').is_file()
print('Base model:', MODEL_PATH)
print('Adapter:', ADAPTER_PATH)
print('Rank:', rank)


Base model: /kaggle/input/models/google/gemma-4/other/gemma-4-e4b-it-qat-w4a16-ct/2
Adapter: /kaggle/input/datasets/michi883/ple-proj-svd-r128
Rank: 128


In [3]:
# 3. Start competition-patched vLLM with LoRA enabled. One server serves BOTH base and adapter.
# Use the wheel installed in cell 1, not a pip-downloaded vLLM.
import os, sys, subprocess, time, requests, torch
from pathlib import Path

assert torch.cuda.is_available(), 'Enable Kaggle GPU acceleration.'
tp_size = min(4, torch.cuda.device_count())
BASE_URL = 'http://127.0.0.1:8000/v1'
LOG_PATH = Path('/tmp/copy_test_lora_vllm.log')

# Stop the server from earlier cells / reruns to prevent duplicate GPU allocations.
if 'lora_server' in globals() and lora_server.poll() is None:
    lora_server.terminate()
    try: lora_server.wait(timeout=30)
    except subprocess.TimeoutExpired: lora_server.kill(); lora_server.wait()
if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

cmd = [sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
       '--model', str(MODEL_PATH), '--served-model-name', MODEL_NAME,
       '--host', '127.0.0.1', '--port', '8000',
       '--tensor-parallel-size', str(tp_size),
       '--max-model-len', '32768', '--gpu-memory-utilization', '0.90',
       '--dtype', 'bfloat16' if torch.cuda.is_bf16_supported() else 'auto',
       '--tool-call-parser', 'gemma4', '--reasoning-parser', 'gemma4',
       '--enable-auto-tool-choice',
       '--enable-lora', '--max-loras', '1', '--max-lora-rank', '128',
       '--lora-modules', f'ple={ADAPTER_PATH}']
print('Starting vLLM, tensor parallel =', tp_size)
print('Adapter alias = ple')
log_handle = open(LOG_PATH, 'w')
lora_server = subprocess.Popen(cmd, stdout=log_handle, stderr=subprocess.STDOUT, env=os.environ.copy())

started = time.monotonic()
while time.monotonic()-started < 20*60:
    if lora_server.poll() is not None:
        print(LOG_PATH.read_text()[-8000:]); raise RuntimeError('vLLM exited during startup')
    try:
        response = requests.get(BASE_URL + '/models', timeout=5)
        if response.ok and response.json().get('data'): break
    except requests.RequestException: pass
    time.sleep(5)
else:
    print(LOG_PATH.read_text()[-8000:]); raise TimeoutError('vLLM not ready after 20 minutes')

ids = [m['id'] for m in response.json()['data']]
print('Served IDs:', ids)
assert MODEL_NAME in ids, 'Base model missing from /models'
assert 'ple' in ids, 'Adapter alias ple missing from /models: check LoRA load logs'
log_tail = LOG_PATH.read_text()
for phrase in ["not in the model's supported LoRA target modules", 'not in supported LoRA target modules']:
    assert phrase not in log_tail, 'LoRA module rejected by vLLM: ' + phrase
print('Ready. Logs:', LOG_PATH)


Starting vLLM, tensor parallel = 4
Adapter alias = ple
Served IDs: ['gemma-4-e4b-it-qat-w4a16-ct', 'ple']
Ready. Logs: /tmp/copy_test_lora_vllm.log


In [4]:
# 4. Compare identical copy prompts with the BASE and LoRA adapter IDs.
# Deliberately enable_thinking=False (the earlier copy-page default is NOT equivalent).
import requests

INPUTS = [
    'redirects', 'directs', 'allow_redirects', 'resolve_redirects',
    'aligns', 'extracts', 'treats', 'diagrams', 'payloads', 'accepts',
    'indenting', 'detecting', 'subtracting', 'dumping', 'interpoling',
    'Constructs a', 'nPlease', 'forks',
]

def copy_once(model_id, source):
    response = requests.post(BASE_URL + '/chat/completions', json={
        'model': model_id,
        'messages': [{'role':'user','content': 'Copy the following text exactly. Output only the copied text:\n' + source}],
        'chat_template_kwargs': {'enable_thinking': False},
        'temperature': 0, 'seed': 1, 'max_tokens': 128,
    }, timeout=180)
    response.raise_for_status()
    body = response.json()
    choice = body['choices'][0]
    msg = choice['message']
    return {'output': msg.get('content') or '',
            'finish_reason': choice.get('finish_reason'),
            'reasoning_present': bool(msg.get('reasoning') or msg.get('reasoning_content'))}

results=[]
for source in INPUTS:
    a=copy_once(MODEL_NAME, source)
    b=copy_once('ple', source)
    row={'input':source,'base':a,'lora':b,'base_match':a['output']==source,'lora_match':b['output']==source}
    results.append(row)
    print(f"{'MATCH' if row['base_match'] else 'FAIL ':5} -> "
          f"{'MATCH' if row['lora_match'] else 'FAIL ':5} "
          f"input={source!r}, base={a['output']!r}, LoRA={b['output']!r}")


FAIL  -> MATCH input='redirects', base='redirect', LoRA='redirects'
FAIL  -> MATCH input='directs', base='direct', LoRA='directs'
FAIL  -> MATCH input='allow_redirects', base='allow_redirect', LoRA='allow_redirects'
FAIL  -> MATCH input='resolve_redirects', base='resolve_redirect', LoRA='resolve_redirects'
FAIL  -> MATCH input='aligns', base='align', LoRA='aligns'
FAIL  -> MATCH input='extracts', base='extract', LoRA='extracts'
FAIL  -> MATCH input='treats', base='treat', LoRA='treats'
MATCH -> MATCH input='diagrams', base='diagrams', LoRA='diagrams'
FAIL  -> MATCH input='payloads', base='payload', LoRA='payloads'
FAIL  -> MATCH input='accepts', base='accept', LoRA='accepts'
FAIL  -> MATCH input='indenting', base='indent', LoRA='indenting'
FAIL  -> MATCH input='detecting', base='detectng', LoRA='detecting'
FAIL  -> MATCH input='subtracting', base='subtract', LoRA='subtracting'
MATCH -> MATCH input='dumping', base='dumping', LoRA='dumping'
FAIL  -> MATCH input='interpoling', base='inter

In [5]:
# 5. Summarize, save machine-readable evidence, and surface any warning.
import json
from pathlib import Path

summary = {
    'total': len(results),
    'base_failures': sum(not r['base_match'] for r in results),
    'lora_failures': sum(not r['lora_match'] for r in results),
    'fixed': sum(not r['base_match'] and r['lora_match'] for r in results),
    'regressions': sum(r['base_match'] and not r['lora_match'] for r in results),
    'any_reasoning_field': any(r[m]['reasoning_present'] for r in results for m in ['base','lora']),
}
path=Path('/kaggle/working/copy_test_lora_results.json')
path.write_text(json.dumps({'model':MODEL_NAME,'adapter':str(ADAPTER_PATH),
                            'thinking_disabled':True,'temperature':0,'seed':1,
                            'summary':summary,'results':results},indent=2))
print(json.dumps(summary,indent=2))
print('Saved:',path)
if summary['fixed'] == 0:
    print('WARNING: No known base error was repaired. Inspect vLLM server logs and LoRA target support.')
if summary['any_reasoning_field']:
    print('WARNING: A response contained a reasoning field; verify thinking-off behavior in model outputs.')
print('LoRA differences observed:', any(r['base']['output'] != r['lora']['output'] for r in results))
print('Check startup/LoRA logs:', LOG_PATH)


{
  "total": 18,
  "base_failures": 15,
  "lora_failures": 1,
  "fixed": 14,
  "regressions": 0,
  "any_reasoning_field": false
}
Saved: /kaggle/working/copy_test_lora_results.json
LoRA differences observed: True
Check startup/LoRA logs: /tmp/copy_test_lora_vllm.log


In [6]:
# 6. Preserve a reproducible evidence bundle for the research paper.
# This cell does not run new inference. Execute it after the comparison and summary.
from pathlib import Path
import hashlib, json, shutil, zipfile, sys, subprocess, datetime
from importlib import metadata

OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)
EVIDENCE = OUT / 'copy_test_lora_evidence'
EVIDENCE.mkdir(exist_ok=True)

def sha256_file(path, chunk_size=4 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for part in iter(lambda: stream.read(chunk_size), b''):
            h.update(part)
    return h.hexdigest()

# Preserve exactly what was submitted to the API and what came back.
assert len(results) == len(INPUTS), 'Run the paired comparison first.'
assert path.is_file(), 'Run the results-summary cell first.'
shutil.copy2(path, EVIDENCE / 'copy_test_lora_results.json')

# Save the active server log, including adapter load messages and warnings.
if LOG_PATH.is_file():
    shutil.copy2(LOG_PATH, EVIDENCE / 'vllm_server.log')
else:
    print('WARNING: vLLM server log not found:', LOG_PATH)

# Small, independently checkable configuration and adapter fingerprints.
adapter_weights = ADAPTER_PATH / 'adapter_model.safetensors'
adapter_config = ADAPTER_PATH / 'adapter_config.json'
assert adapter_weights.is_file() and adapter_config.is_file()
shutil.copy2(adapter_config, EVIDENCE / 'adapter_config.json')
model_config = MODEL_PATH / 'config.json'
if model_config.is_file():
    shutil.copy2(model_config, EVIDENCE / 'model_config.json')

try:
    installed_vllm = metadata.version('vllm')
except metadata.PackageNotFoundError:
    installed_vllm = 'unknown'

manifest = {
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'experiment': 'Kaggle E4B W4A16 base vs rank-128 LoRA exact-copy test',
    'model_name': MODEL_NAME,
    'model_path': str(MODEL_PATH),
    'model_config_sha256': sha256_file(model_config) if model_config.is_file() else None,
    'adapter_alias': 'ple',
    'adapter_path': str(ADAPTER_PATH),
    'adapter_weights_sha256': sha256_file(adapter_weights),
    'adapter_weights_bytes': adapter_weights.stat().st_size,
    'adapter_config_sha256': sha256_file(adapter_config),
    'adapter_rank': rank,
    'vllm_version': installed_vllm,
    'python_version': sys.version,
    'tensor_parallel_size': tp_size,
    'server_command': cmd,
    'served_model_ids': ids,
    'prompt_template': 'Copy the following text exactly. Output only the copied text:\n' + '{input}',
    'request_settings': {'chat_template_kwargs': {'enable_thinking': False},
                         'temperature': 0, 'seed': 1, 'max_tokens': 128},
    'summary': summary,
    'limitations': ['Targeted examples, not a random evaluation sample.',
                    'A successful adapter load and changed outputs do not alone prove broad coding-agent gains.',
                    'Full base checkpoint hash not calculated because the checkpoint is large.'],
}
(EVIDENCE / 'manifest.json').write_text(json.dumps(manifest, indent=2))
(EVIDENCE / 'README.txt').write_text(
    'Copy Test - LoRA: Kaggle vLLM evidence bundle\n'
    'Run notebook cells in order. This bundle contains exact paired outputs, '
    'model/adapter configuration fingerprints, and the vLLM server log.\n'
    'Compare manifest.json with copy_test_lora_results.json. '
    'A private Kaggle dataset may prevent public reruns until the adapter is published.\n'
)
zip_path = OUT / 'copy_lora_evidence.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(EVIDENCE.iterdir()):
        if file.is_file():
            archive.write(file, arcname=file.name)
with zipfile.ZipFile(zip_path) as archive:
    assert archive.testzip() is None
    print('Evidence bundle:', zip_path)
    print('Files:', archive.namelist())
print('Base failures:', summary['base_failures'],
      '| LoRA failures:', summary['lora_failures'],
      '| Fixed:', summary['fixed'],
      '| Regressions:', summary['regressions'])
print('Adapter SHA256:', manifest['adapter_weights_sha256'])


Evidence bundle: /kaggle/working/copy_lora_evidence.zip
Files: ['README.txt', 'adapter_config.json', 'copy_test_lora_results.json', 'manifest.json', 'model_config.json', 'vllm_server.log']
Base failures: 15 | LoRA failures: 1 | Fixed: 14 | Regressions: 0
Adapter SHA256: 9ce83a7cad3d8523c14750696dca06ee1ca89f50813047ae390ee00c7f17c888


### Interpretation

A **base failure → LoRA success** under the same vLLM server is the critical observation missing from the earlier merged-weight simulation. If neither model fails, this run does not establish improvement. If the adapter is listed by `/models` but nothing changes, inspect logs for unsupported-module warnings and confirm its projection wrapper is actually used.

This is **an inference test, not a Kaggle submission or an agent-edit benchmark**. On successful validation, expand the fixed list to the saved 273 failing words and then test coding-agent edits.
